# Sankey diagrams for target-species detections

This notebook creates one Sankey/alluvial diagram per barcode from the exported `barcodeXX_target_species_detection.tsv` tables inside `data/mergedseq5`.

Flow direction:

```text
expected_forward_primer → detected_forward_primer → detected_species → target_species
```

The notebook also expands the target-detection table so expected forward primers, SaproxDNA codes and detected forward primers are no longer semicolon-separated lists inside one cell. The expanded table receives trust annotations and is exported as:

```text
barcodeXX_target_species_detection_trusted.tsv
```

Two Sankey figures are written per barcode:

```text
barcodeXX_target_detection_sankey.png
barcodeXX_target_detection_sankey_trusted.png
```

The first keeps the source-based color scheme. The second colors trusted links in green hues and untrusted links in grey.


A later section reuses the same plotting and trust rules for the filtered below-50% target-species table:

```text
barcodeXX_target_species_taxids_below_50pct_forward_primers.tsv
```

That section does not export new tables; it only writes Sankey figures for the filtered rows.


## Trust rules

Trust is assigned after expanding list-like primer/code cells into one expected-forward-primer/detected-forward-primer combination per row.

Primer-detected rows:

- For each `target_species + expected_forward_primer` pair, a row is trusted only when `expected_forward_primer == detected_forward_primer`.
- If a target/species expected-primer pair has multiple detected forward primers, only the matching detected primer is trusted; mismatching detected primers are not trusted.
- If a target species has two or more expected forward primers, rows are evaluated after expansion. Each expected/detected primer combination is trusted only when the expanded expected and detected forward primers match.

No-forward-primer rows:

- If `detected_forward_primer` is `none`, primer matching cannot be used.
- Such rows are trusted only when `target_genus` and `taxid_detected_genus` match and the detected species is unique to one `target_species + expected_forward_primer` pair in that barcode.
- If the same detected species is linked to multiple target/species expected-primer pairs, the no-forward-primer row is not trusted.

Missing values:

- Missing text values such as blank strings, `nan`, `<NA>`, `none`, and `null` are treated as `none`.
- Rows with no usable read-count weight are not drawn in the Sankey, but remain represented in the expanded trusted table.

The trust classification is conservative. It is intended to identify rows suitable for exploratory follow-up, not to make final taxonomic decisions.


In [ ]:
from pathlib import Path
import os
import re
import textwrap

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch, Rectangle, Patch
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data/mergedseq5'
SANKEY_NOTEBOOK_VERSION = '20260717_target_expected_detected_trust_v2'
print(f'Sankey notebook version: {SANKEY_NOTEBOOK_VERSION}')

# Use None to run every barcode table found in data/mergedseq5/barcode*/.
# Example subset: SELECTED_BARCODES = ['barcode01', 'barcode06']
SELECTED_BARCODES = None

FIGURE_DPI = 250
FIGURE_WIDTH = 18
MIN_FIGURE_HEIGHT = 8
NODE_WIDTH = 0.055
LINK_ALPHA = 0.72
LINK_EDGE_WIDTH = 0.25
LINK_CENTERLINE_WIDTH = 0.45
LINK_CENTERLINE_ALPHA = 0.90
LABEL_MAX_CHARS = 42

# Optional readability controls. Keep as None to retain all categories.
MAX_TARGET_SPECIES = None
MAX_EXPECTED_FORWARD_PRIMERS = None
MAX_DETECTED_FORWARD_PRIMERS = None
MAX_DETECTED_SPECIES = None


## General helpers

These functions standardize missing values, split semicolon-separated cells and keep natural primer ordering such that `bc2` sorts before `bc10`.


In [ ]:
def clean_category(value, missing_label='none'):
    if pd.isna(value):
        return missing_label
    value = re.sub('[​-‍﻿]', '', str(value)).strip()
    if value == '' or value.lower() in {'nan', '<na>', 'none', 'null'}:
        return missing_label
    return value


def normalize_key(value):
    value = clean_category(value, missing_label='none')
    value = value.lower()
    value = re.sub(r'\s+', '_', value)
    value = re.sub(r'_+', '_', value)
    return value.strip('_') or 'none'


def split_semicolon_values(value):
    if pd.isna(value):
        return []
    values = []
    for item in str(value).split(';'):
        item = clean_category(item, missing_label='')
        if item:
            values.append(item)
    return values


def natural_sort_key(value):
    value = str(value)
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', value)]


def shorten_label(value, width=LABEL_MAX_CHARS):
    value = str(value)
    if len(value) <= width:
        return value
    return textwrap.shorten(value, width=width, placeholder='…')


## Read, expand and classify target-detection rows

The expansion keeps `target_sxdcodes` aligned with `expected_forward`. If both lists have the same length, values are paired positionally. If one list has a single value and the other has multiple, the single value is repeated. If both have multiple values but different lengths, all combinations are used and the row is flagged in `expansion_note`.


In [ ]:
def barcode_table_paths(data_dir=DATA_DIR, selected_barcodes=SELECTED_BARCODES):
    paths = sorted(data_dir.glob('barcode*/barcode*_target_species_detection.tsv'))
    if selected_barcodes is None:
        return paths
    selected = set(selected_barcodes)
    return [path for path in paths if path.parent.name in selected]


def read_detection_table(path):
    table = pd.read_csv(path, sep='	', dtype='string')
    required = {'target_species', 'target_genus', 'target_sxdcodes', 'expected_forward',
        'forward_primer', 'taxid_detected_name', 'taxid_detected_genus'}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f'{path} is missing required columns: {sorted(missing)}')
    return table


def pair_sxdcode_expected_forward(sxdcodes_value, expected_forward_value):
    sxdcodes = split_semicolon_values(sxdcodes_value) or ['none']
    expected_forwards = split_semicolon_values(expected_forward_value) or ['none']
    if len(sxdcodes) == len(expected_forwards):
        return list(zip(sxdcodes, expected_forwards)), 'paired_by_position'
    if len(sxdcodes) == 1:
        return [(sxdcodes[0], expected_forward) for expected_forward in expected_forwards], 'repeated_single_sxdcode'
    if len(expected_forwards) == 1:
        return [(sxdcode, expected_forwards[0]) for sxdcode in sxdcodes], 'repeated_single_expected_forward'
    return [(sxdcode, expected_forward) for sxdcode in sxdcodes
        for expected_forward in expected_forwards], 'all_combinations_length_mismatch'


def row_weight(row):
    if clean_category(row.get('detected_forward_primer'), missing_label='none') != 'none':
        candidates = ['taxid_primer_read_count', 'taxid_read_count', 'target_species_read_count']
    else:
        candidates = ['no_forward_primer_read_count', 'taxid_read_count', 'target_species_read_count']
    for column in candidates:
        if column in row.index and pd.notna(row[column]):
            value = pd.to_numeric(row[column], errors='coerce')
            if pd.notna(value) and value > 0:
                return float(value)
    return 0.0


In [ ]:
def expand_detection_table(table):
    rows = []
    for input_index, row in table.reset_index(drop=False).iterrows():
        species_values = split_semicolon_values(row.get('target_species')) or ['none']
        sxd_expected_pairs, expansion_note = pair_sxdcode_expected_forward(
            row.get('target_sxdcodes'), row.get('expected_forward'))
        detected_forwards = split_semicolon_values(row.get('forward_primer')) or ['none']
        denominator = max(len(species_values) * len(sxd_expected_pairs) * len(detected_forwards), 1)
        weight = row_weight(row)
        for target_species in species_values:
            for target_sxdcode, expected_forward in sxd_expected_pairs:
                for detected_forward in detected_forwards:
                    record = row.to_dict()
                    record['source_row_index'] = input_index
                    record['target_species_original'] = row.get('target_species')
                    record['target_sxdcodes_original'] = row.get('target_sxdcodes')
                    record['expected_forward_original'] = row.get('expected_forward')
                    record['forward_primer_original'] = row.get('forward_primer')
                    record['target_species'] = clean_category(target_species)
                    record['target_sxdcode'] = clean_category(target_sxdcode)
                    record['expected_forward_primer'] = clean_category(expected_forward)
                    record['detected_forward_primer'] = clean_category(detected_forward)
                    record['detected_species'] = clean_category(row.get('taxid_detected_name'))
                    record['detected_genus'] = clean_category(row.get('taxid_detected_genus'))
                    record['target_genus_clean'] = clean_category(row.get('target_genus'))
                    record['expansion_note'] = expansion_note
                    record['sankey_read_count'] = weight / denominator if weight > 0 else 0.0
                    rows.append(record)
    if not rows:
        return pd.DataFrame(columns=list(table.columns) + [
            'source_row_index', 'target_species_original', 'target_sxdcode',
            'expected_forward_primer', 'detected_forward_primer', 'detected_species',
            'detected_genus', 'target_genus_clean', 'expansion_note', 'sankey_read_count'])
    return pd.DataFrame(rows)


In [ ]:
def add_trust_classification(expanded):
    expanded = expanded.copy()
    expanded['target_expected_pair_key'] = (expanded['target_species'].map(normalize_key)
        + '|' + expanded['expected_forward_primer'].map(normalize_key))
    expanded['detected_species_key'] = expanded['detected_species'].map(normalize_key)
    expanded['target_genus_key'] = expanded['target_genus_clean'].map(normalize_key)
    expanded['detected_genus_key'] = expanded['detected_genus'].map(normalize_key)
    detected_species_pair_counts = (expanded[expanded['detected_species_key'].ne('none')]
        .drop_duplicates(['detected_species_key', 'target_expected_pair_key'])
        .groupby('detected_species_key')['target_expected_pair_key']
        .nunique()
        .to_dict())

    trusted_values = []
    trust_reasons = []
    for row in expanded.itertuples(index=False):
        expected = normalize_key(row.expected_forward_primer)
        detected = normalize_key(row.detected_forward_primer)
        detected_species = normalize_key(row.detected_species)
        target_genus = normalize_key(row.target_genus_clean)
        detected_genus = normalize_key(row.detected_genus)
        if detected != 'none':
            if expected != 'none' and expected == detected:
                trusted_values.append(True)
                trust_reasons.append('detected_forward_matches_expected_forward')
            else:
                trusted_values.append(False)
                trust_reasons.append('detected_forward_mismatches_expected_forward')
        else:
            species_pair_count = detected_species_pair_counts.get(detected_species, 0)
            if (target_genus != 'none' and target_genus == detected_genus
                    and detected_species != 'none' and species_pair_count == 1):
                trusted_values.append(True)
                trust_reasons.append('no_forward_detected_target_genus_matches_unique_detected_species')
            elif target_genus == 'none' or detected_genus == 'none':
                trusted_values.append(False)
                trust_reasons.append('no_forward_detected_missing_target_or_detected_genus')
            elif target_genus != detected_genus:
                trusted_values.append(False)
                trust_reasons.append('no_forward_detected_genus_mismatch')
            elif detected_species == 'none':
                trusted_values.append(False)
                trust_reasons.append('no_forward_detected_missing_detected_species')
            else:
                trusted_values.append(False)
                trust_reasons.append('no_forward_detected_detected_species_not_unique_to_target_expected_pair')
    expanded['trusted'] = trusted_values
    expanded['trust_reason'] = trust_reasons
    return expanded


## Sankey flow preparation

The Sankey uses the expanded, trust-classified table. Link weights come from `sankey_read_count`, which is derived from the read-count columns in the input table and split across expansions so the total flow is conserved.


In [ ]:
def collapse_low_frequency(values, weights, max_categories, other_label='other'):
    if max_categories is None:
        return values
    totals = pd.DataFrame({'value': values, 'weight': weights}).groupby('value')['weight'].sum()
    keep = set(totals.sort_values(ascending=False).head(max_categories).index)
    return [value if value in keep else other_label for value in values]


def build_sankey_flows(expanded):
    columns = ['target_species', 'expected_forward_primer', 'detected_forward_primer',
        'detected_species', 'trusted', 'sankey_read_count']
    source = expanded[columns].copy()
    source = source[source['sankey_read_count'] > 0].copy()
    if source.empty:
        return pd.DataFrame(columns=['target_species', 'expected_forward_primer',
            'detected_forward_primer', 'detected_species', 'trusted', 'read_count'])
    source['target_species'] = collapse_low_frequency(
        source['target_species'], source['sankey_read_count'], MAX_TARGET_SPECIES, other_label='other_target_species')
    source['expected_forward_primer'] = collapse_low_frequency(
        source['expected_forward_primer'], source['sankey_read_count'], MAX_EXPECTED_FORWARD_PRIMERS, other_label='other_expected_forward')
    source['detected_forward_primer'] = collapse_low_frequency(
        source['detected_forward_primer'], source['sankey_read_count'], MAX_DETECTED_FORWARD_PRIMERS, other_label='other_detected_forward')
    source['detected_species'] = collapse_low_frequency(
        source['detected_species'], source['sankey_read_count'], MAX_DETECTED_SPECIES, other_label='other_detected_species')
    return (source.groupby(['target_species', 'expected_forward_primer', 'detected_forward_primer',
            'detected_species', 'trusted'], as_index=False)
        .agg(read_count=('sankey_read_count', 'sum'))
        .sort_values('read_count', ascending=False))


## Plotting helpers

The plotting code builds a four-stage alluvial diagram directly in Matplotlib. Thin flows are drawn with high opacity and a narrow centerline to make small links easier to trace. This is visual-only; link weights still use the original read-count-derived values.

Layer order is optimized with repeated weighted barycentric sweeps for primer layers only. Detected and target species layers stay alphabetically ordered. This does not eliminate all crossings, but it usually reduces path overlap substantially compared with count-only ordering.


In [ ]:
def make_links(flows, source_column, target_column, keep_trust=False):
    group_columns = [source_column, target_column]
    if keep_trust:
        group_columns.append('trusted')
    return (flows.groupby(group_columns, as_index=False)
        .agg(read_count=('read_count', 'sum'))
        .rename(columns={source_column: 'source', target_column: 'target'}))


def node_order(values, totals, sort_mode='count'):
    if sort_mode == 'natural':
        return sorted(values, key=lambda value: (value == 'none', natural_sort_key(value)))
    if sort_mode == 'alphabetical':
        return sorted(values, key=lambda value: (value == 'none', str(value).lower()))
    return sorted(values, key=lambda value: (value == 'none', -totals.get(value, 0), str(value)))


def layer_layout(nodes, totals, total_flow, gap_fraction=0.012):
    if not nodes:
        return {}
    gap = max(total_flow * gap_fraction, total_flow * 0.002)
    y = 0.0
    layout = {}
    for node in nodes:
        height = max(float(totals.get(node, 0)), total_flow * 0.002)
        layout[node] = {'y0': y, 'y1': y + height, 'height': height}
        y += height + gap
    return layout


def ribbon_path(x0, y0a, y0b, x1, y1a, y1b, curvature=0.45):
    dx = x1 - x0
    c0 = x0 + dx * curvature
    c1 = x1 - dx * curvature
    vertices = [
        (x0, y0a), (c0, y0a), (c1, y1a), (x1, y1a),
        (x1, y1b), (c1, y1b), (c0, y0b), (x0, y0b),
        (x0, y0a)]
    codes = [
        MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.CLOSEPOLY]
    return MplPath(vertices, codes)


def centerline_path(x0, y0, x1, y1, curvature=0.45):
    dx = x1 - x0
    c0 = x0 + dx * curvature
    c1 = x1 - dx * curvature
    vertices = [(x0, y0), (c0, y0), (c1, y1), (x1, y1)]
    codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4]
    return MplPath(vertices, codes)



def draw_link_layer(ax, links, source_layout, target_layout, x_source, x_target,
        color_by_source=None, color_by_trust=False):
    source_offsets = {node: values['y0'] for node, values in source_layout.items()}
    target_offsets = {node: values['y0'] for node, values in target_layout.items()}
    sort_columns = ['source', 'target'] + (['trusted'] if 'trusted' in links.columns else []) + ['read_count']
    ascending = [True, True] + ([False] if 'trusted' in links.columns else []) + [False]
    links = links.sort_values(sort_columns, ascending=ascending)
    for row in links.itertuples(index=False):
        weight = float(row.read_count)
        if weight <= 0:
            continue
        y0a = source_offsets[row.source]
        y0b = y0a + weight
        y1a = target_offsets[row.target]
        y1b = y1a + weight
        source_offsets[row.source] = y0b
        target_offsets[row.target] = y1b
        if color_by_trust:
            color = '#1b9e77' if getattr(row, 'trusted', False) else '#bdbdbd'
            alpha = 0.82 if getattr(row, 'trusted', False) else 0.34
        else:
            color = color_by_source.get(row.source, '#808080') if color_by_source is not None else '#808080'
            alpha = LINK_ALPHA
        path = ribbon_path(x_source + NODE_WIDTH / 2, y0a, y0b, x_target - NODE_WIDTH / 2, y1a, y1b)
        patch = PathPatch(path, facecolor=color, edgecolor=color,
            linewidth=LINK_EDGE_WIDTH, alpha=alpha, zorder=1)
        ax.add_patch(patch)
        line_path = centerline_path(x_source + NODE_WIDTH / 2, (y0a + y0b) / 2,
            x_target - NODE_WIDTH / 2, (y1a + y1b) / 2)
        line = PathPatch(line_path, facecolor='none', edgecolor=color,
            linewidth=LINK_CENTERLINE_WIDTH, alpha=LINK_CENTERLINE_ALPHA, zorder=2)
        ax.add_patch(line)


def draw_nodes(ax, layer_nodes, layer_layouts, x_positions, layer_titles):
    for layer_index, nodes in enumerate(layer_nodes):
        x = x_positions[layer_index]
        for node in nodes:
            values = layer_layouts[layer_index][node]
            rect = Rectangle((x - NODE_WIDTH / 2, values['y0']), NODE_WIDTH,
                values['height'], facecolor='white', edgecolor='black', linewidth=0.7, zorder=3)
            ax.add_patch(rect)
            y_mid = (values['y0'] + values['y1']) / 2
            ha = 'right' if layer_index == 0 else 'left'
            label_x = x - NODE_WIDTH if layer_index == 0 else x + NODE_WIDTH
            ax.text(label_x, y_mid, shorten_label(node), ha=ha, va='center', fontsize=7, zorder=4)
        ax.text(x, -max(v['height'] for layout in layer_layouts for v in layout.values()) * 0.04,
            layer_titles[layer_index], ha='center', va='bottom', fontsize=11, fontweight='bold')


In [ ]:
def links_for_totals(flows, layer_columns):
    return [make_links(flows, layer_columns[index], layer_columns[index + 1])
        for index in range(len(layer_columns) - 1)]


def layer_totals_from_links(links_by_layer, layer_columns):
    layer_totals = []
    for index, column in enumerate(layer_columns):
        incoming = (links_by_layer[index - 1].groupby('target')['read_count'].sum()
            if index > 0 else pd.Series(dtype='float64'))
        outgoing = (links_by_layer[index].groupby('source')['read_count'].sum()
            if index < len(links_by_layer) else pd.Series(dtype='float64'))
        layer_totals.append(pd.concat([incoming, outgoing], axis=1).max(axis=1).to_dict())
    return layer_totals


def initial_layer_nodes(layer_totals):
    return [
        node_order(layer_totals[0].keys(), layer_totals[0], sort_mode='natural'),
        node_order(layer_totals[1].keys(), layer_totals[1], sort_mode='natural'),
        node_order(layer_totals[2].keys(), layer_totals[2], sort_mode='alphabetical'),
        node_order(layer_totals[3].keys(), layer_totals[3], sort_mode='alphabetical')]


def weighted_neighbor_position(links, node_column, neighbor_column, neighbor_positions):
    positions = {}
    grouped = links.groupby(node_column)
    for node, group in grouped:
        total = group['read_count'].sum()
        if total > 0:
            positions[node] = sum(group['read_count'] * group[neighbor_column].map(neighbor_positions)) / total
    return positions


def reorder_layer_by_barycenter(nodes, barycenters, totals):
    return sorted(nodes, key=lambda node: (
        node == 'none',
        barycenters.get(node, float('inf')),
        -totals.get(node, 0),
        natural_sort_key(node)))


def minimise_crossing_order(layer_nodes, links_by_layer, layer_totals, movable_layers=None, passes=8):
    # Weighted barycentric sweeps. Species layers are intentionally fixed
    # alphabetically; only primer layers are allowed to move.
    if movable_layers is None:
        movable_layers = {0, 1}
    layer_nodes = [list(nodes) for nodes in layer_nodes]
    for _ in range(passes):
        for layer_index in range(1, len(layer_nodes)):
            if layer_index not in movable_layers:
                continue
            previous_positions = {node: position for position, node in enumerate(layer_nodes[layer_index - 1])}
            barycenters = weighted_neighbor_position(
                links_by_layer[layer_index - 1], 'target', 'source', previous_positions)
            layer_nodes[layer_index] = reorder_layer_by_barycenter(
                layer_nodes[layer_index], barycenters, layer_totals[layer_index])
        for layer_index in range(len(layer_nodes) - 2, -1, -1):
            if layer_index not in movable_layers:
                continue
            next_positions = {node: position for position, node in enumerate(layer_nodes[layer_index + 1])}
            barycenters = weighted_neighbor_position(
                links_by_layer[layer_index], 'source', 'target', next_positions)
            layer_nodes[layer_index] = reorder_layer_by_barycenter(
                layer_nodes[layer_index], barycenters, layer_totals[layer_index])
    return layer_nodes


def sankey_layout_inputs(flows):
    layer_columns = ['expected_forward_primer', 'detected_forward_primer',
        'detected_species', 'target_species']
    layer_titles = ['Expected forward primer', 'Detected forward primer',
        'Detected species', 'Target species']
    links_by_layer = links_for_totals(flows, layer_columns)
    total_flow = float(flows['read_count'].sum())
    layer_totals = layer_totals_from_links(links_by_layer, layer_columns)
    layer_nodes = initial_layer_nodes(layer_totals)
    layer_nodes = minimise_crossing_order(layer_nodes, links_by_layer, layer_totals, movable_layers={0, 1})
    layer_layouts = [layer_layout(nodes, totals, total_flow) for nodes, totals in zip(layer_nodes, layer_totals)]
    return layer_columns, layer_titles, links_by_layer, layer_nodes, layer_layouts, total_flow


def plot_sankey(flows, barcode, output_path=None, color_trusted=False):
    if flows.empty:
        print(f'{barcode}: no rows with read-count evidence; no Sankey diagram created')
        return None
    layer_columns, layer_titles, links_by_layer, layer_nodes, layer_layouts, total_flow = sankey_layout_inputs(flows)
    max_extent = max(max(layout[node]['y1'] for node in layout) if layout else 0 for layout in layer_layouts)
    max_node_count = max(len(nodes) for nodes in layer_nodes)
    fig_height = max(MIN_FIGURE_HEIGHT, 0.18 * max_node_count)
    fig, ax = plt.subplots(figsize=(FIGURE_WIDTH, fig_height))

    cmap = colormaps['tab20']
    color_by_expected_forward = {node: cmap(index % cmap.N) for index, node in enumerate(layer_nodes[0])}
    color_by_detected_forward = {node: cmap(index % cmap.N) for index, node in enumerate(layer_nodes[1])}
    color_by_detected_species = {node: cmap(index % cmap.N) for index, node in enumerate(layer_nodes[2])}
    layer_colors = [color_by_expected_forward, color_by_detected_forward, color_by_detected_species]

    x_positions = list(range(len(layer_columns)))
    trusted_links_by_layer = [make_links(flows, layer_columns[index], layer_columns[index + 1], keep_trust=True)
        for index in range(len(layer_columns) - 1)]
    for index, links in enumerate(trusted_links_by_layer if color_trusted else links_by_layer):
        draw_link_layer(ax, links, layer_layouts[index], layer_layouts[index + 1],
            x_positions[index], x_positions[index + 1], layer_colors[index], color_by_trust=color_trusted)
    draw_nodes(ax, layer_nodes, layer_layouts, x_positions, layer_titles)

    if color_trusted:
        legend_handles = [Patch(facecolor='#1b9e77', label='trusted'),
            Patch(facecolor='#bdbdbd', label='not trusted')]
        ax.legend(handles=legend_handles, loc='upper right', title='Trust status')
    ax.set_xlim(-0.65, len(layer_columns) - 1 + 0.75)
    ax.set_ylim(max_extent * 1.03, -max_extent * 0.08)
    suffix = ' trusted links highlighted' if color_trusted else ''
    ax.set_title(f'{barcode}: expected primer → detected primer → detected species → target species{suffix}', fontsize=13)
    ax.axis('off')
    plt.tight_layout()

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=FIGURE_DPI, bbox_inches='tight')
        print(f'Saved {output_path}')
    return fig


## Build expanded tables and Sankey diagrams

This cell loops over the selected barcode tables. For each barcode it:

1. Reads `barcodeXX_target_species_detection.tsv`.
2. Expands semicolon-separated expected primers, SaproxDNA codes and detected forward primers into one combination per row.
3. Adds `trusted` and `trust_reason` columns.
4. Exports `barcodeXX_target_species_detection_trusted.tsv`.
5. Saves a standard Sankey and a trusted-highlight Sankey.


In [ ]:
summary_rows = []

for table_path in barcode_table_paths():
    barcode = table_path.parent.name
    print(f'Processing {barcode}: {table_path}')
    table = read_detection_table(table_path)
    expanded = add_trust_classification(expand_detection_table(table))
    trusted_output = table_path.parent / f'{barcode}_target_species_detection_trusted.tsv'
    expanded.to_csv(trusted_output, sep='	', index=False)
    print(f'Exported {len(expanded):,} expanded rows to {trusted_output}')

    flows = build_sankey_flows(expanded)
    output_path = table_path.parent / 'figures' / f'{barcode}_target_detection_sankey.png'
    trusted_output_path = table_path.parent / 'figures' / f'{barcode}_target_detection_sankey_trusted.png'
    fig = plot_sankey(flows, barcode, output_path=output_path, color_trusted=False)
    if fig is not None:
        plt.show()
    trusted_fig = plot_sankey(flows, barcode, output_path=trusted_output_path, color_trusted=True)
    if trusted_fig is not None:
        plt.show()
    summary_rows.append({'barcode': barcode,
        'input_rows': len(table),
        'expanded_rows': len(expanded),
        'trusted_rows': int(expanded['trusted'].sum()) if len(expanded) else 0,
        'flow_rows': len(flows),
        'total_flow_read_count': flows['read_count'].sum() if not flows.empty else 0,
        'unique_target_species': flows['target_species'].nunique() if not flows.empty else 0,
        'unique_expected_forward_primers': flows['expected_forward_primer'].nunique() if not flows.empty else 0,
        'unique_detected_forward_primers': flows['detected_forward_primer'].nunique() if not flows.empty else 0,
        'unique_detected_species': flows['detected_species'].nunique() if not flows.empty else 0,
        'trusted_table': str(trusted_output),
        'figure': str(output_path) if fig is not None else pd.NA,
        'trusted_figure': str(trusted_output_path) if trusted_fig is not None else pd.NA})

sankey_summary = pd.DataFrame(summary_rows)
display(sankey_summary)


## Sankey diagrams for below-50% target-species detections

This section reads the filtered tables:

```text
data/mergedseq5/barcodeXX/barcodeXX_target_species_taxids_below_50pct_forward_primers.tsv
```

It applies the same expansion, trust classification and Sankey plotting rules used above. No new expanded/trusted tables are exported from this section. Only figures are written.

Output figures:

```text
barcodeXX/figures/barcodeXX_target_species_taxids_below_50pct_forward_primers_sankey.png
barcodeXX/figures/barcodeXX_target_species_taxids_below_50pct_forward_primers_sankey_trusted.png
```


In [ ]:
def below_50pct_table_paths(data_dir=DATA_DIR, selected_barcodes=SELECTED_BARCODES):
    paths = sorted(data_dir.glob('barcode*/barcode*_target_species_taxids_below_50pct_forward_primers.*'))
    paths = [path for path in paths if path.suffix.lower() in {'.tsv', '.csv', '.txt'}]
    if selected_barcodes is None:
        return paths
    selected = set(selected_barcodes)
    return [path for path in paths if path.parent.name in selected]


below_50pct_summary_rows = []

for table_path in below_50pct_table_paths():
    barcode = table_path.parent.name
    print(f'Processing below-50% table for {barcode}: {table_path}')
    table = read_detection_table(table_path)
    expanded = add_trust_classification(expand_detection_table(table))
    flows = build_sankey_flows(expanded)

    output_path = (table_path.parent / 'figures'
        / f'{barcode}_target_species_taxids_below_50pct_forward_primers_sankey.png')
    trusted_output_path = (table_path.parent / 'figures'
        / f'{barcode}_target_species_taxids_below_50pct_forward_primers_sankey_trusted.png')

    fig = plot_sankey(flows, f'{barcode} below-50%', output_path=output_path, color_trusted=False)
    if fig is not None:
        plt.show()
    trusted_fig = plot_sankey(flows, f'{barcode} below-50%',
        output_path=trusted_output_path, color_trusted=True)
    if trusted_fig is not None:
        plt.show()

    below_50pct_summary_rows.append({'barcode': barcode,
        'input_rows': len(table),
        'expanded_rows': len(expanded),
        'trusted_rows': int(expanded['trusted'].sum()) if len(expanded) else 0,
        'flow_rows': len(flows),
        'total_flow_read_count': flows['read_count'].sum() if not flows.empty else 0,
        'unique_target_species': flows['target_species'].nunique() if not flows.empty else 0,
        'unique_expected_forward_primers': flows['expected_forward_primer'].nunique() if not flows.empty else 0,
        'unique_detected_forward_primers': flows['detected_forward_primer'].nunique() if not flows.empty else 0,
        'unique_detected_species': flows['detected_species'].nunique() if not flows.empty else 0,
        'figure': str(output_path) if fig is not None else pd.NA,
        'trusted_figure': str(trusted_output_path) if trusted_fig is not None else pd.NA})

below_50pct_sankey_summary = pd.DataFrame(below_50pct_summary_rows)
display(below_50pct_sankey_summary)


## Optional: inspect one barcode's expanded Sankey input table

Use this cell when a diagram looks unexpected. It shows the expanded trust-classified table and the aggregated flow table used for one barcode.


In [ ]:
INSPECT_BARCODE = 'barcode01'
inspect_path = DATA_DIR / INSPECT_BARCODE / f'{INSPECT_BARCODE}_target_species_detection.tsv'
inspect_table = read_detection_table(inspect_path)
inspect_expanded = add_trust_classification(expand_detection_table(inspect_table))
inspect_flows = build_sankey_flows(inspect_expanded)
display(inspect_expanded[['target_species', 'target_sxdcode', 'expected_forward_primer',
    'detected_forward_primer', 'detected_species', 'target_genus_clean', 'detected_genus',
    'sankey_read_count', 'trusted', 'trust_reason']].head(50))
display(inspect_flows.sort_values('read_count', ascending=False).head(50))


## Export one combined trusted table across barcodes

This final section combines trusted rows from all selected barcode target-species detection tables into one mergedseq5-level table.

Output:

```text
data/mergedseq5/mergedseq5_target_species_detection_trusted.tsv
```

Only rows where `trusted == True` are retained. If a per-barcode `barcodeXX_target_species_detection_trusted.tsv` file already exists, it is loaded. If it does not exist yet, the table is rebuilt in memory from `barcodeXX_target_species_detection.tsv` using the same expansion and trust rules as above.


In [ ]:
COMBINED_TRUSTED_OUTPUT = DATA_DIR / 'mergedseq5_target_species_detection_trusted.tsv'

combined_trusted_tables = []
for table_path in barcode_table_paths():
    barcode = table_path.parent.name
    trusted_path = table_path.parent / f'{barcode}_target_species_detection_trusted.tsv'
    if trusted_path.exists():
        trusted_table = pd.read_csv(trusted_path, sep='	', dtype='string')
        if 'trusted' in trusted_table.columns:
            trusted_table['trusted'] = trusted_table['trusted'].astype('string').str.lower().eq('true')
    else:
        source_table = read_detection_table(table_path)
        trusted_table = add_trust_classification(expand_detection_table(source_table))
    if trusted_table.empty or 'trusted' not in trusted_table.columns:
        continue
    trusted_only = trusted_table[trusted_table['trusted'] == True].copy()
    if trusted_only.empty:
        continue
    trusted_only.insert(0, 'barcode', barcode)
    trusted_only.insert(1, 'source_target_species_detection_file', str(table_path))
    combined_trusted_tables.append(trusted_only)

if combined_trusted_tables:
    combined_trusted = pd.concat(combined_trusted_tables, ignore_index=True, sort=False)
else:
    combined_trusted = pd.DataFrame()

combined_trusted.to_csv(COMBINED_TRUSTED_OUTPUT, sep='	', index=False)
print(f'Exported {len(combined_trusted):,} trusted rows to {COMBINED_TRUSTED_OUTPUT}')
display(combined_trusted)
